# Evaluation on Implicit Hate (Explicit Hate Filtered)

This notebook evaluates all saved models (baseline from `weights/` and RAG from `weights_rag/`) on test sets where **explicit hate examples are removed**, focusing on the harder implicit hate detection task.

Each model is evaluated on the dataset it was trained on.

> **Note for RAG models:** these were fine-tuned on retrieval-augmented inputs but are evaluated here on **plain text** — the same basis as baseline models — to allow fair comparison. This is intentional for cross-model comparison; it will underestimate RAG model performance relative to their augmented-input setting.

## 1. Imports

In [1]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from datasets import load_dataset, Dataset
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
import warnings
warnings.filterwarnings('ignore')

## 2. Configuration

In [2]:
ROOT_DIR        = Path('.')
WEIGHTS_DIR     = ROOT_DIR / 'weights'
WEIGHTS_RAG_DIR = ROOT_DIR / 'weights_rag'

MAX_LENGTH = 256
BATCH_SIZE = 32

# ISHate: value(s) in the 'implicit_layer' column that indicate *explicit* hate
# Run the dataset cell first — it prints all unique values so you can verify this.
ISHATE_EXPLICIT_LABELS = {'Explicit HS'}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


## 3. Load & Filter Datasets

- **IHC**: filter out `class == 'explicit_hate'` from the test split
- **ISHate**: filter out rows where `implicit_layer` indicates explicit hate (see `ISHATE_EXPLICIT_LABELS`)

In [3]:
# ── IHC ──────────────────────────────────────────────────────────────────────
raw_ihc = load_dataset('tasksource/implicit-hate-stg1', split='train')
splits  = raw_ihc.train_test_split(test_size=0.10, seed=42)
test_ihc_full = splits['test']
test_ihc = test_ihc_full.filter(lambda x: x['class'] != 'explicit_hate')
test_ihc = test_ihc.map(lambda x: {'label': 0 if x['class'] == 'not_hate' else 1})

print(f'IHC test — full: {len(test_ihc_full):,}  after filtering explicit: {len(test_ihc):,}')

# ── ISHate ────────────────────────────────────────────────────────────────────
ishate_raw = load_dataset('BenjaminOcampo/ISHate')
test_ishate_full = ishate_raw['test']

print(f"\nISHate 'implicit_layer' unique values: {set(test_ishate_full['implicit_layer'])}")
print(f"  → filtering out: {ISHATE_EXPLICIT_LABELS}")

test_ishate = test_ishate_full.filter(lambda x: x['implicit_layer'] not in ISHATE_EXPLICIT_LABELS)
test_ishate = test_ishate.map(lambda x: {'label': 0 if x['hateful_layer'] == 'Non-HS' else 1})

print(f'ISHate test — full: {len(test_ishate_full):,}  after filtering explicit: {len(test_ishate):,}')

# ── Registry ──────────────────────────────────────────────────────────────────
TEST_SETS = {
    'IHC':    {'dataset': test_ihc,    'text_col': 'post'},
    'ISHate': {'dataset': test_ishate, 'text_col': 'text'},
}

IHC test — full: 2,148  after filtering explicit: 2,028

ISHate 'implicit_layer' unique values: {'Implicit HS', None, 'Explicit HS'}
  → filtering out: {'Explicit HS'}
ISHate test — full: 4,368  after filtering explicit: 2,867


## 4. Model Registry

6 RAG models: `{model}/base/{dataset}` from `weights_rag/`. Evaluated on plain text (no retrieval augmentation).

In [4]:
# RAG models: weights_rag/{model}/base/{dataset}
RAG_MODELS = [
    {'path': WEIGHTS_RAG_DIR / 'bert'     / 'base' / 'IHC',    'dataset': 'IHC',    'label': 'bert (RAG) / IHC'},
    {'path': WEIGHTS_RAG_DIR / 'bert'     / 'base' / 'ISHate', 'dataset': 'ISHate', 'label': 'bert (RAG) / ISHate'},
    {'path': WEIGHTS_RAG_DIR / 'hatebert' / 'base' / 'IHC',    'dataset': 'IHC',    'label': 'hatebert (RAG) / IHC'},
    {'path': WEIGHTS_RAG_DIR / 'hatebert' / 'base' / 'ISHate', 'dataset': 'ISHate', 'label': 'hatebert (RAG) / ISHate'},
    {'path': WEIGHTS_RAG_DIR / 'roberta'  / 'base' / 'IHC',    'dataset': 'IHC',    'label': 'roberta (RAG) / IHC'},
    {'path': WEIGHTS_RAG_DIR / 'roberta'  / 'base' / 'ISHate', 'dataset': 'ISHate', 'label': 'roberta (RAG) / ISHate'},
]

all_models = RAG_MODELS

# Check which models actually have weights on disk
print(f"{'Model':<40} {'Dataset':<12} Weights?")
print('-' * 68)
for m in all_models:
    has_weights = (m['path'] / 'model.safetensors').exists() or (m['path'] / 'pytorch_model.bin').exists()
    status = '✓' if has_weights else '✗  (missing)'
    print(f"{m['label']:<40} {m['dataset']:<12} {status}")

Model                                    Dataset      Weights?
--------------------------------------------------------------------
bert (RAG) / IHC                         IHC          ✓
bert (RAG) / ISHate                      ISHate       ✓
hatebert (RAG) / IHC                     IHC          ✓
hatebert (RAG) / ISHate                  ISHate       ✓
roberta (RAG) / IHC                      IHC          ✓
roberta (RAG) / ISHate                   ISHate       ✓


## 5. Helpers

In [5]:
def tokenize_plain(hf_dataset, text_col, tokenizer):
    texts  = list(hf_dataset[text_col])
    labels = list(hf_dataset['label'])
    encoded = tokenizer(
        texts,
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
    )
    encoded['labels'] = labels
    return Dataset.from_dict(encoded)


def compute_metrics(eval_pred):
    preds  = np.argmax(eval_pred.predictions, axis=-1)
    labels = eval_pred.label_ids
    return {
        'macro_f1': f1_score(labels, preds, average='macro',  zero_division=0),
        'macro_p':  precision_score(labels, preds, average='macro', zero_division=0),
        'macro_r':  recall_score(labels, preds, average='macro',    zero_division=0),
    }

## 6. Evaluation Loop

In [6]:
results = {}

eval_args = TrainingArguments(
    output_dir='./tmp_eval',
    per_device_eval_batch_size=BATCH_SIZE,
    report_to='none',
)

for entry in all_models:
    has_weights = (entry['path'] / 'model.safetensors').exists() or (entry['path'] / 'pytorch_model.bin').exists()
    if not has_weights:
        print(f"[skip] {entry['label']} — no weights on disk")
        continue

    ds_name = entry['dataset']
    if ds_name not in TEST_SETS:
        print(f"[skip] {entry['label']} — no test set for {ds_name}")
        continue

    print(f"\n{'='*60}")
    print(f"{entry['label']}  →  {ds_name} (implicit only)")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(entry['path'])
    model     = AutoModelForSequenceClassification.from_pretrained(entry['path'])

    tok_test = tokenize_plain(
        TEST_SETS[ds_name]['dataset'],
        TEST_SETS[ds_name]['text_col'],
        tokenizer,
    )

    trainer = Trainer(
        model=model,
        args=eval_args,
        compute_metrics=compute_metrics,
    )

    preds_out = trainer.predict(tok_test)
    preds  = np.argmax(preds_out.predictions, axis=-1)
    labels = list(TEST_SETS[ds_name]['dataset']['label'])

    print(classification_report(labels, preds, target_names=['Non-HS', 'HS']))

    results[entry['label']] = {
        'dataset':  ds_name,
        'macro_f1': f1_score(labels, preds, average='macro',  zero_division=0),
        'macro_p':  precision_score(labels, preds, average='macro', zero_division=0),
        'macro_r':  recall_score(labels, preds, average='macro',    zero_division=0),
    }

    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()


bert (RAG) / IHC  →  IHC (implicit only)


  0%|          | 0/64 [00:00<?, ?it/s]

              precision    recall  f1-score   support

      Non-HS       0.84      0.86      0.85      1330
          HS       0.72      0.69      0.71       698

    accuracy                           0.80      2028
   macro avg       0.78      0.78      0.78      2028
weighted avg       0.80      0.80      0.80      2028


bert (RAG) / ISHate  →  ISHate (implicit only)


  0%|          | 0/90 [00:00<?, ?it/s]

              precision    recall  f1-score   support

      Non-HS       0.98      0.90      0.94      2681
          HS       0.37      0.80      0.50       186

    accuracy                           0.90      2867
   macro avg       0.68      0.85      0.72      2867
weighted avg       0.94      0.90      0.91      2867


hatebert (RAG) / IHC  →  IHC (implicit only)


...

              precision    recall  f1-score   support

      Non-HS       0.98      0.90      0.94      2681
          HS       0.36      0.78      0.50       186

    accuracy                           0.90      2867
   macro avg       0.67      0.84      0.72      2867
weighted avg       0.94      0.90      0.91      2867


roberta (RAG) / IHC  →  IHC (implicit only)


  0%|          | 0/64 [00:00<?, ?it/s]

              precision    recall  f1-score   support

      Non-HS       0.82      0.90      0.86      1330
          HS       0.76      0.63      0.69       698

    accuracy                           0.81      2028
   macro avg       0.79      0.76      0.77      2028
weighted avg       0.80      0.81      0.80      2028


roberta (RAG) / ISHate  →  ISHate (implicit only)


  0%|          | 0/90 [00:00<?, ?it/s]

              precision    recall  f1-score   support

      Non-HS       0.97      0.95      0.96      2681
          HS       0.42      0.58      0.49       186

    accuracy                           0.92      2867
   macro avg       0.70      0.76      0.72      2867
weighted avg       0.93      0.92      0.93      2867



## 7. Results

One table per dataset. Rows = model, columns = Macro F1 / Precision / Recall.

In [7]:
for ds_name in TEST_SETS:
    ds_results = {k: v for k, v in results.items() if v['dataset'] == ds_name}
    if not ds_results:
        continue
    df = pd.DataFrame({
        k: {'Macro F1': v['macro_f1'], 'Macro Precision': v['macro_p'], 'Macro Recall': v['macro_r']}
        for k, v in ds_results.items()
    }).T
    df.index.name = 'Model'
    styled = (
        df.style
        .format('{:.3f}')
        .highlight_max(axis=0, props='font-weight: bold; background-color: #d4f1d4')
        .set_caption(f'Implicit-only evaluation — {ds_name}')
    )
    display(styled)

,Macro F1,Macro Precision,Macro Recall
Model,,,
bert (RAG) / IHC,0.780,0.783,0.776
hatebert (RAG) / IHC,0.770,0.774,0.766
roberta (RAG) / IHC,0.774,0.793,0.763


,Macro F1,Macro Precision,Macro Recall
Model,,,
bert (RAG) / ISHate,0.724,0.676,0.853
hatebert (RAG) / ISHate,0.720,0.674,0.845
roberta (RAG) / ISHate,0.723,0.697,0.761
